# **LLM Math Chain**

In [ ]:
!pip install langchain==0.0.300

In [ ]:
!pip install langchain langchain_openai

In [20]:
import openai
import os
from langchain.chat_models import ChatOpenAI # Correct import for langchain==0.0.300
from langchain.prompts.chat import ChatPromptTemplate, HumanMessagePromptTemplate # Correct import for langchain==0.0.300
from langchain.schema import HumanMessage # Correct import for langchain==0.0.300
from langchain.chains import LLMMathChain
from langchain.llms import OpenAI
from langchain.chains import LLMChain, SequentialChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

In [ ]:
os.environ['OPENAI_API_KEY'] = ''

openai.api_key = os.getenv('OPENAI_API_KEY')

llm = ChatOpenAI()

In [6]:
result = llm([HumanMessage(content="What is 20 + 15 - 10")])

result.content


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `__call__` was deprecated in LangChain 0.1.7 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


'25'

In [7]:
result = llm([HumanMessage(content="What is 200345 * 181")])

result.content

'36,149,145'

# **LLMRouter Chain**

In [10]:
hr_template = """You are a very smart human resources person. \
You are great at answering questions about human resources in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""

In [11]:
law_template = """You are a very smart lawyer. \
You are great at answering questions about the law in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""


In [12]:
prompt_infos = [
    {
        'name': 'human resources',
        'description': 'Answer human resources questions',
        'template': hr_template
    },
    {
        'name': 'law',
        'description': 'Answers legal questions',
        'template': law_template
    }
]

In [13]:
destination_chains = {}

In [ ]:
# Create destination chains
for p_info in prompt_infos:
    name = p_info['name']
    prompt_template = p_info['template']
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain


In [ ]:
# Default chain
default_prompt = ChatPromptTemplate.from_template('{input}')
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
# View destinations
destinations


# Join into single string
destination_str = "\n".join(destinations)
print(destination_str)


# Format router template
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destination_str
)

router_template

In [ ]:
print(MULTI_PROMPT_ROUTER_TEMPLATE)